In [ ]:
import os
import json

import networkx as nx
from WordNodes import *
import nltk
from nltk.corpus import wordnet as wn
from tqdm import tqdm

from helpers import nltk_to_wordnet, tokenize_1, replace_contractions, wordnet_pos, get_correct_pos_tag

c:\Users\nsmit\Desktop\Projects\LING581-Natural-Language-Processing-BYU\Final_Project\helpers.py:91: SyntaxWarning: invalid escape sequence '\W'
  pattern = r'(^|\W)(' + contractions + ')($|\W)'


In [4]:
def len_of_split(string):
    a = string.split('_')
    return len(a)

In [5]:
def preprocess_text(text):
     return replace_contractions(text)

In [6]:
def get_wordnet_usable_tokens(pos_tagged_tokens):
    wordnet_usable_tokens = set()
    for token, tag in pos_tagged_tokens:
        if tag in nltk_to_wordnet.keys():
            wordnet_tag = nltk_to_wordnet[tag]
            # if pos_tag matches a definition
            if len(wn.synsets(token, pos=wordnet_tag[0])) > 0 or (wordnet_tag[0] == 'a' and len(wn.synsets(token, pos='s')) > 0):
                wordnet_usable_tokens.add((token.lower(), wordnet_tag))

            # otherwise give an unknown pos_tag
            else:
                all_synsets = wn.synsets(token)
                if len(all_synsets) > 0:
                    first_pos = all_synsets[0].pos()
                    if all([syn.pos() == first_pos for syn in all_synsets]):       # if there is only one pos, assume that that is the correct one
                        wordnet_usable_tokens.add((token.lower(), wordnet_pos[first_pos]))
                    else:
                        wordnet_usable_tokens.add((token.lower(), ("UNK_POS",)))

    return wordnet_usable_tokens
        

In [7]:
def verify_pos_tag(word, nltk_assigned_pos_tag):
    if nltk_assigned_pos_tag not in nltk_to_wordnet:
        return False
    
    valid_pos_tags = set([synset.pos() for synset in wn.synsets(word)])
    pos = nltk_to_wordnet[nltk_assigned_pos_tag]

    if pos not in valid_pos_tags:
        return False
        
    return (word, pos)

### Get List of Tokens

In [3]:
example_sentences = []
for synset in wn.all_synsets():
    for sentence in synset.examples():
        example_sentences.append(sentence)

In [6]:
def get_tokens_from_corpus(files=None):
    all_tokens = set()
    file_list = [txt_file for txt_file in os.listdir('Corpus') if '.txt' in txt_file] if files is None else files
    loop = tqdm(total=len(file_list))
    for txt_file in file_list:
        # open file and replace all contractions with expanded form
        loop.set_description(f'Total Tokens: {len(all_tokens)}; processing {txt_file} - opening file...')
        with open(os.path.join('Corpus', txt_file), 'r', encoding='utf-8') as file:
            text = preprocess_text(file.read())
    
        # get tokens that have a synset in wordnet (to get a better list of all forms of different words)
        loop.set_description(f'Total Tokens: {len(all_tokens)}; processing {txt_file} - tokenizing...')
        corpus_tokens = tokenize_1(text)                 # tokenize corpus

        loop.set_description(f'Total Tokens: {len(all_tokens)}; processing {txt_file} - pos_tagging...')
        corpus_tagged_tokens = nltk.pos_tag(corpus_tokens)    # tokenize_corpus
        
        corpus_tokens = get_wordnet_usable_tokens(corpus_tagged_tokens)
        
        all_tokens = all_tokens.union(corpus_tokens)
        loop.update()

    loop.set_description(f'Total Tokens: {len(all_tokens)}; complete.')
    loop.close()

    return all_tokens
    

In [8]:
def get_network_tokens(corpus_files=None):
    tokens = set()
    
    # tokenize and process wordnet definitions
    for synset in tqdm(wn.all_synsets(), 'processing wordnet definitions...'):
        # tag individual lemmas
        lemmas_pos_tagged = set()
        lemmas = synset.lemmas()
        for lemma in lemmas:
            lemma_pos_tag = get_correct_pos_tag(lemma.name(), synset.pos())
            
            lemmas_pos_tagged = lemmas_pos_tagged.union([(lemma.name().lower(), lemma_pos_tag)])

        # tag example sentences to get tokens
        examples_pos_tagged = set()
        for sentence in synset.examples():
            sent_tokenized = tokenize_1(sentence)
            for lemma in [l.name() for l in lemmas if l.name() in sent_tokenized]:
                for l2 in lemmas:
                    new_sent = [(word if word != lemma else l2.name()) for word in sent_tokenized]
                    example_pos_tagged = nltk.pos_tag(new_sent)
                    example_tokens = get_wordnet_usable_tokens(example_pos_tagged)
                    examples_pos_tagged = examples_pos_tagged.union(example_tokens)

        # tag definitions to get tokens
        definition = tokenize_1(preprocess_text(synset.definition()))
        def_pos_tagged = set(get_wordnet_usable_tokens(nltk.pos_tag(definition)))
        
        tokens = tokens.union((def_pos_tagged.union(lemmas_pos_tagged).union(examples_pos_tagged)))
                           
    # add tokens from corpus
    tokens = tokens.union(get_tokens_from_corpus(files=corpus_files))
    
    return tokens


In [9]:
tokens = get_network_tokens()

processing wordnet definitions...: 117659it [2:11:44, 14.88it/s]
Total Tokens: 147269; complete.: 100%|██████████████████████████████████████████████████| 8/8 [48:26<00:00, 363.26s/it]


In [10]:
tokens_list = list(tokens)

In [11]:
tokens_set = set(tokens_list)

In [12]:
tokens_set

{('hayrick', ('n', 'singular', 'common')),
 ('pastoral', ('n', 'singular', 'common')),
 ('babylonian_captivity', ('n', 'singular', 'common')),
 ('interdicted', ('v', 'base')),
 ('lading', ('UNK_POS',)),
 ('decarboxylate', ('v', 'base')),
 ('ganesh', ('n', 'singular', 'common')),
 ('stereoscopy', ('n', 'singular', 'common')),
 ('christian_liturgy', ('n', 'singular', 'proper')),
 ('apostrophe', ('n',)),
 ('concerning', ('v', 'base')),
 ('idlers', ('n', 'plural', 'common')),
 ('systematic_desensitisation', ('n', 'singular', 'common')),
 ('shingle', ('n', 'singular', 'proper')),
 ('observer', ('n',)),
 ('blunt_trauma', ('n', 'singular', 'common')),
 ('neglected', ('v', 'past', 'participle')),
 ('mosaicism', ('n', 'singular', 'common')),
 ('incarnated', ('v', 'past')),
 ('vapourous', ('a',)),
 ('wetter', ('n', 'singular', 'proper')),
 ('dostoevskian', ('a',)),
 ('fingers', ('v', 'present', '3rd-person', 'singular')),
 ('sniffle', ('v', 'base')),
 ('haws', ('n', 'singular', 'proper')),
 ('cu

In [13]:
file_path = 'tokens_raw.json'

In [14]:
with open(file_path, 'w') as file:
    json.dump(tokens_list, file)

### Make Graph

In [8]:
file_path = 'tokens_raw.json'

In [9]:
def read_in_raw_tokens(file_path=file_path):
    with open(file_path, 'r') as file:
        tokens_json = json.load(file)
        
    for i in range(len(tokens_json)):
        token = tokens_json[i]
        token[1] = tuple(token[1])
        tokens_json[i] = tuple(tokens_json[i])
        
    tokens_set = set(tokens_json)
    return tokens_set
    

In [10]:
tokens_set = read_in_raw_tokens()

In [11]:
meta_nodes = {'singular': MetaNode('singular', opposites=['plural']),
              'plural': MetaNode('plural', opposites=['singular']),
              'common': MetaNode('common', opposites=['proper']),
              'proper': MetaNode('proper', opposites=['common']),
              'base': MetaNode('base', opposites=['participle']),
              'past': MetaNode('past', opposites=['present']),
              'present': MetaNode('present', opposites=['past']),
              '3rd-person': MetaNode('3rd-person', opposites=['non-3rd-person']),
              'non-3rd-person': MetaNode('non-3rd-person', opposites=['3rd-person']),
              'participle': MetaNode('participle', opposites=['base']),
              'comparative': MetaNode('comparative', opposites=['superlative']),
              'superlative': MetaNode('superlative', opposites=['comparative'])}

In [21]:
nodes = {}
for token in tqdm(tokens_set, 'creating nodes...'):
    word, meta = token
    pos = meta[0]
    
    node = WordNode(token)

    if pos != "UNK_POS":
        for syn in wn.synsets(word, pos=pos):
            node.add_definition(syn)
    nodes[token] = node
            

creating nodes...: 100%|████████████████████████████████████████████████████| 264311/264311 [00:03<00:00, 81594.35it/s]


In [22]:
graph = nx.DiGraph()
for key, meta_node in meta_nodes.items():
    graph.add_node(meta_node, label=(key, 'meta'), value=key, type='meta')
    
for key, node in tqdm(nodes.items(), 'creating graph...'):
    word, pos = key
    pos, *meta = pos
    
    graph.add_node(node, label=key, value=word, type='word')
    for meta_node_label in meta:
        graph.add_edge(node, meta_nodes[meta_node_label], weight=1)
        
    for synset, definition in node.definitions.items():
        definition_tokens = get_wordnet_usable_tokens(nltk.pos_tag(tokenize_1(definition)))
        for tok in definition_tokens:
            other = nodes[tok]
            if not graph.has_edge(node, other):
                graph.add_edge(node, other, weight=1)
            else:
                graph[node][other]['weight'] += 1
    
        
        

creating graph...: 100%|██████████████████████████████████████████████████████| 264311/264311 [12:44<00:00, 345.84it/s]


In [42]:
nx.write_gexf(graph, 'pos_tagged_graph_original.gexf')

### Update Edge Weights

In [43]:
graph = nx.read_gexf('pos_tagged_graph_original.gexf')

In [44]:
graph_copy = graph.copy()

In [45]:
# remove selfloops
count = 0
for node in tqdm(graph_copy.nodes(), 'removing_selfloops...'):
    if graph_copy.has_edge(node, node):
        graph_copy.remove_edge(node, node)
        count += 1
print(f'{count} selfloops found and removed')

removing_selfloops...: 100%|██████████████████████████████████████████████| 264323/264323 [00:00<00:00, 1183223.69it/s]

0 selfloops found and removed


In [46]:
# weight each edge proportional to the in-degree of the node it leads to 
for node in tqdm(graph_copy.nodes(), 'updating_weights...'):
    in_edge_attributes = [edge[2] for edge in graph_copy.in_edges(node, data=True)]
    in_degree = sum([item['weight'] for item in in_edge_attributes])
    for edge_attributes in in_edge_attributes:
        edge_attributes['weight'] = edge_attributes['weight'] / in_degree
    

updating_weights...: 100%|██████████████████████████████████████████████████| 264323/264323 [00:04<00:00, 54891.44it/s]


In [47]:
# convert weights on edges to probabilities
regularization = 0.005
for node in tqdm(graph_copy.nodes(), 'calculating probabilities...'):
    out_edge_attributes = [edge[2] for edge in graph_copy.out_edges(node, data=True)]
    total_out = sum([(item['weight']**2 + regularization) for item in out_edge_attributes])
    for edge_attribute in out_edge_attributes:
        edge_attribute['weight'] = (edge_attribute['weight']**2 + regularization) / total_out

calculating probabilities...: 100%|█████████████████████████████████████████| 264323/264323 [00:04<00:00, 60029.91it/s]


In [48]:
nx.write_gexf(graph_copy, 'graph_weighted_with_probs.gexf')